# Advanced Forecasting Models

## 1. Introduction

This notebook extends the Bitcoin forecasting benchmark beyond the earlier classical, LSTM, and vanilla Transformer experiments. It keeps the same daily Bitcoin Close target and the same 80/20 chronological train-test split so model scores remain comparable across notebooks.

Models that cannot run in the current environment are documented explicitly as **Not run in current environment**. No metrics are invented for unavailable packages.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling1D,
    Input,
    LayerNormalization,
    MultiHeadAttention,
)
from tensorflow.keras.models import Model, Sequential

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.metrics import mae, rmse, mape, smape

warnings.filterwarnings("ignore")
tf.random.set_seed(42)
np.random.seed(42)

pd.set_option("display.float_format", "{:.6f}".format)

## 2. Load Dataset

In [ ]:
data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"

raw_df = load_bitcoin_data(data_path)
df_daily = prepare_daily_bitcoin_data(raw_df)
target = df_daily["Close"].asfreq("D").dropna().rename("Close")

target.head(), target.tail(), target.shape

## 3. Train-Test Split

In [ ]:
split_idx = int(len(target) * 0.8)
train = target.iloc[:split_idx]
test = target.iloc[split_idx:]
y_test = test.copy()

print("Train period:", train.index.min(), "to", train.index.max())
print("Test period:", test.index.min(), "to", test.index.max())
print("Train shape:", train.shape)
print("Test shape:", test.shape)

## 4. Classical Extensions

In [ ]:
def evaluate_forecast(y_true, y_pred):
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
        "sMAPE": smape(y_true, y_pred),
    }


forecasts = {}
model_status = {}

naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
moving_average_forecast = (
    target.shift(1)
    .rolling(window=7)
    .mean()
    .reindex(test.index)
    .rename("7-Day Moving Average")
)

forecasts["Naive"] = naive_forecast
forecasts["7-Day Moving Average"] = moving_average_forecast
model_status["Naive"] = "Run"
model_status["7-Day Moving Average"] = "Run"

arima_model = SARIMAX(
    train,
    order=(1, 1, 1),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
arima_results = arima_model.fit(disp=False)
arima_forecast = arima_results.forecast(steps=len(test))
arima_forecast.index = test.index
arima_forecast = arima_forecast.rename("ARIMA(1,1,1)")

forecasts["ARIMA(1,1,1)"] = arima_forecast
model_status["ARIMA(1,1,1)"] = "Run"

pd.DataFrame(
    [evaluate_forecast(y_test, forecast) for forecast in forecasts.values()],
    index=forecasts.keys(),
).sort_values("RMSE")

## 5. SARIMA

In [ ]:
sarima_model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_results = sarima_model.fit(disp=False)
sarima_forecast = sarima_results.forecast(steps=len(test))
sarima_forecast.index = test.index
sarima_forecast = sarima_forecast.rename("SARIMA")

forecasts["SARIMA"] = sarima_forecast
model_status["SARIMA"] = "Run: SARIMAX(order=(1,1,1), seasonal_order=(1,0,1,7))"

pd.DataFrame([evaluate_forecast(y_test, sarima_forecast)], index=["SARIMA"])

## 6. Prophet

In [ ]:
prophet_available = importlib.util.find_spec("prophet") is not None

if prophet_available:
    from prophet import Prophet

    prophet_train = train.reset_index()
    prophet_train.columns = ["ds", "y"]
    prophet_train["ds"] = prophet_train["ds"].dt.tz_localize(None)

    prophet_model = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=True,
        daily_seasonality=False,
    )
    prophet_model.fit(prophet_train)

    future = pd.DataFrame({"ds": test.index.tz_localize(None)})
    prophet_predictions = prophet_model.predict(future)
    prophet_forecast = pd.Series(
        prophet_predictions["yhat"].to_numpy(),
        index=test.index,
        name="Prophet",
    )

    forecasts["Prophet"] = prophet_forecast
    model_status["Prophet"] = "Run"
    display(pd.DataFrame([evaluate_forecast(y_test, prophet_forecast)], index=["Prophet"]))
else:
    model_status["Prophet"] = "Not run in current environment: package `prophet` is not installed. Install with `pip install prophet`."
    print(model_status["Prophet"])

## 7. Transformer-Based Models

In [ ]:
LOOKBACK = 30

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train.to_numpy().reshape(-1, 1))
test_scaled = scaler.transform(test.to_numpy().reshape(-1, 1))


def create_sequences(values, lookback):
    X, y = [], []
    for i in range(lookback, len(values)):
        X.append(values[i - lookback : i])
        y.append(values[i])
    return np.array(X), np.array(y)


X_train, y_train = create_sequences(train_scaled, LOOKBACK)
combined_scaled = np.vstack([train_scaled[-LOOKBACK:], test_scaled])
X_test, _ = create_sequences(combined_scaled, LOOKBACK)


def fit_lstm_forecast(lookback, layers, name):
    X_train_lstm, y_train_lstm = create_sequences(train_scaled, lookback)
    combined_lstm = np.vstack([train_scaled[-lookback:], test_scaled])
    X_test_lstm, _ = create_sequences(combined_lstm, lookback)

    model = Sequential(layers)
    model.compile(optimizer="adam", loss="mse")
    callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    model.fit(
        X_train_lstm,
        y_train_lstm,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        callbacks=[callback],
        shuffle=False,
        verbose=0,
    )
    predictions_scaled = model.predict(X_test_lstm, verbose=0)
    predictions = scaler.inverse_transform(predictions_scaled).ravel()
    return pd.Series(predictions, index=test.index, name=name)


original_lstm_forecast = fit_lstm_forecast(
    lookback=30,
    layers=[
        Input(shape=(30, 1)),
        tf.keras.layers.LSTM(32),
        Dense(1),
    ],
    name="Original LSTM",
)

improved_lstm_forecast = fit_lstm_forecast(
    lookback=60,
    layers=[
        Input(shape=(60, 1)),
        tf.keras.layers.LSTM(64, return_sequences=True),
        Dropout(0.2),
        tf.keras.layers.LSTM(32),
        Dropout(0.2),
        Dense(1),
    ],
    name="Improved LSTM",
)

forecasts["Original LSTM"] = original_lstm_forecast
forecasts["Improved LSTM"] = improved_lstm_forecast
model_status["Original LSTM"] = "Run"
model_status["Improved LSTM"] = "Run"


def corrected_transformer_encoder(inputs, d_model=64, num_heads=4, ff_dim=128, dropout=0.1):
    attention_output = MultiHeadAttention(
        key_dim=d_model // num_heads,
        num_heads=num_heads,
        dropout=dropout,
    )(inputs, inputs)
    attention_output = Dropout(dropout)(attention_output)
    x = LayerNormalization(epsilon=1e-6)(inputs + attention_output)

    feed_forward = Dense(ff_dim, activation="relu")(x)
    feed_forward = Dropout(dropout)(feed_forward)
    feed_forward = Dense(d_model)(feed_forward)
    return LayerNormalization(epsilon=1e-6)(x + feed_forward)


D_MODEL = 64
corrected_inputs = Input(shape=(LOOKBACK, 1))
x = Dense(D_MODEL)(corrected_inputs)
x = corrected_transformer_encoder(x, d_model=D_MODEL, num_heads=4, ff_dim=128, dropout=0.1)
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(32, activation="relu")(x)
corrected_outputs = Dense(1)(x)

corrected_transformer_model = Model(corrected_inputs, corrected_outputs)
corrected_transformer_model.compile(optimizer="adam", loss="mse")
corrected_callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
corrected_history = corrected_transformer_model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[corrected_callback],
    shuffle=False,
    verbose=0,
)

corrected_predictions_scaled = corrected_transformer_model.predict(X_test, verbose=0)
corrected_predictions = scaler.inverse_transform(corrected_predictions_scaled).ravel()
corrected_transformer_forecast = pd.Series(
    corrected_predictions,
    index=test.index,
    name="Corrected Transformer",
)

forecasts["Corrected Transformer"] = corrected_transformer_forecast
model_status["Corrected Transformer"] = "Run: d_model=64 corrected encoder"

pd.DataFrame(
    [evaluate_forecast(y_test, forecasts[name]) for name in ["Original LSTM", "Improved LSTM", "Corrected Transformer"]],
    index=["Original LSTM", "Improved LSTM", "Corrected Transformer"],
).sort_values("RMSE")

## 8. PatchTST Workflow

In [ ]:
patchtst_available = importlib.util.find_spec("neuralforecast") is not None

if patchtst_available:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import PatchTST

    nf_train = pd.DataFrame(
        {
            "unique_id": "BTC",
            "ds": train.index.tz_localize(None),
            "y": train.to_numpy(),
        }
    )

    patchtst_model = PatchTST(
        h=len(test),
        input_size=LOOKBACK,
        max_steps=100,
    )
    patchtst_nf = NeuralForecast(models=[patchtst_model], freq="D")
    patchtst_nf.fit(df=nf_train)
    patchtst_predictions = patchtst_nf.predict()

    patchtst_column = "PatchTST"
    patchtst_forecast = pd.Series(
        patchtst_predictions[patchtst_column].to_numpy(),
        index=test.index,
        name="PatchTST",
    )

    forecasts["PatchTST"] = patchtst_forecast
    model_status["PatchTST"] = "Run via neuralforecast"
    display(pd.DataFrame([evaluate_forecast(y_test, patchtst_forecast)], index=["PatchTST"]))
else:
    model_status["PatchTST"] = "Not run in current environment: no usable PatchTST package found. A common option is `neuralforecast` with `PatchTST`."
    print(model_status["PatchTST"])

## 9. iTransformer Workflow

In [ ]:
itransformer_available = importlib.util.find_spec("neuralforecast") is not None

if itransformer_available:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import iTransformer

    nf_train = pd.DataFrame(
        {
            "unique_id": "BTC",
            "ds": train.index.tz_localize(None),
            "y": train.to_numpy(),
        }
    )

    itransformer_model = iTransformer(
        h=len(test),
        input_size=LOOKBACK,
        max_steps=100,
    )
    itransformer_nf = NeuralForecast(models=[itransformer_model], freq="D")
    itransformer_nf.fit(df=nf_train)
    itransformer_predictions = itransformer_nf.predict()

    itransformer_column = "iTransformer"
    itransformer_forecast = pd.Series(
        itransformer_predictions[itransformer_column].to_numpy(),
        index=test.index,
        name="iTransformer",
    )

    forecasts["iTransformer"] = itransformer_forecast
    model_status["iTransformer"] = "Run via neuralforecast"
    display(pd.DataFrame([evaluate_forecast(y_test, itransformer_forecast)], index=["iTransformer"]))
else:
    model_status["iTransformer"] = "Not run in current environment: no usable iTransformer package found. A common option is `neuralforecast` with `iTransformer`."
    print(model_status["iTransformer"])

## 10. Optional Informer / Autoformer Notes

Informer and Autoformer are useful references for long-horizon Transformer-style forecasting, but they are not implemented here unless a compatible local package is already available and easy to use. In this environment, the required libraries are not assumed. Add them only after deciding on a specific framework and validating its data format, horizon handling, and reproducibility controls.

Suggested future workflow:

1. Choose a maintained implementation, such as a forecasting library that exposes Informer/Autoformer models.
2. Convert the Bitcoin Close series into the required long-format dataset.
3. Use the same 80/20 chronological split and 30-day input context where possible.
4. Report metrics only after the model actually runs locally.

## 11. Evaluation Metrics

In [ ]:
metrics_table = pd.DataFrame(
    [evaluate_forecast(y_test, forecast) for forecast in forecasts.values()],
    index=forecasts.keys(),
).sort_values("RMSE")

model_status_table = pd.DataFrame.from_dict(
    model_status,
    orient="index",
    columns=["Status"],
)

print("Model run status:")
display(model_status_table)

print("Evaluation metrics for models that ran:")
metrics_table

## 12. Model Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y_test.plot(ax=ax, label="Actual", linewidth=2)

plot_models = [
    "Naive",
    "7-Day Moving Average",
    "ARIMA(1,1,1)",
    "SARIMA",
    "Corrected Transformer",
]

for model_name in plot_models:
    if model_name in forecasts:
        forecasts[model_name].plot(ax=ax, label=model_name, alpha=0.85)

ax.set_title("Advanced Forecasting Benchmark")
ax.set_xlabel("Date")
ax.set_ylabel("Bitcoin Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

metrics_table

## 13. Key Findings

This notebook is designed to separate runnable local benchmarks from placeholder workflows. SARIMA and ARIMA extend the classical baseline family with statsmodels. Prophet, PatchTST, and iTransformer are included as environment-gated workflows: if their packages are unavailable, they are marked as **Not run in current environment** and excluded from the metrics table.

The most important benchmark remains the naive one-step persistence forecast. Any advanced model should be judged against that baseline before being treated as useful for Bitcoin Close forecasting.